In [7]:
# Importation des librairies
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

In [2]:
# Importation du dataset
df = pd.read_csv("../data/rent_prediction.csv")
df.head()

,IdentifiantMaison,Chambres,Salon,SalleDeBainInterieure,Parking,Meuble,Jardin,Superficie_m2,DistanceRoute_m,Quartier,AgeMaison,LoyerMensuel_BIF
0,1,4.0,Oui,Oui,NaN,Non,Oui,192.0,277.0,Rohero,17.0,2600000
1,2,5.0,Oui,Oui,Non,Oui,Non,234.0,47.0,Rohero,37.0,2600000
2,3,3.0,Oui,Oui,Non,Non,Oui,151.0,45.0,Kinanira,23.0,1069218
3,4,5.0,Oui,Oui,Oui,Oui,Oui,227.0,174.0,Gasekebuye,22.0,2600000
4,5,5.0,Oui,Oui,Oui,NaN,Non,262.0,216.0,Gihosha,21.0,2085458


In [3]:
df = df.drop(columns=["IdentifiantMaison"], axis=1)
df.head()

,Chambres,Salon,SalleDeBainInterieure,Parking,Meuble,Jardin,Superficie_m2,DistanceRoute_m,Quartier,AgeMaison,LoyerMensuel_BIF
0,4.0,Oui,Oui,NaN,Non,Oui,192.0,277.0,Rohero,17.0,2600000
1,5.0,Oui,Oui,Non,Oui,Non,234.0,47.0,Rohero,37.0,2600000
2,3.0,Oui,Oui,Non,Non,Oui,151.0,45.0,Kinanira,23.0,1069218
3,5.0,Oui,Oui,Oui,Oui,Oui,227.0,174.0,Gasekebuye,22.0,2600000
4,5.0,Oui,Oui,Oui,NaN,Non,262.0,216.0,Gihosha,21.0,2085458


# Traitement des valeurs manquantes

Les valeurs manquantes vont être imputer plus tard dans le création un pipeline, en
calculant les statistiques uniquement sur le train.

# Encodage des variables catégorielles

In [4]:
# Encodage de la variable Quartier

# Regrouper les quartiers peu fréquents en "Autres"
counts = df["Quartier"].value_counts()
rare_quarters = counts[counts < 25].index

df["Quartier"] = df["Quartier"].replace(rare_quarters, "Autres")

# Stockage du dataset d'entrainement du modele
df.to_csv("../data/processed_dataset.csv", index=False)

La variable **Quartier** étant nominale, elle sera encodée par **One-Hot Encoding** dans la création d'un pipeline afin d’éviter toute hiérarchie artificielle entre les catégories. Les quartiers faiblement représentés ont été regroupés dans une modalité **Autres** avant l’encodage, ce qui limite le risque de surapprentissage et améliore la robustesse du modèle.

# Scaling des variables numériques 

Les variables numériques (Superficie_m2, DistanceRoute_m, AgeMaison, Chambres et Confort_Score) présentent des échelles de mesure différentes. Un processus de normalisation est donc appliqué afin de garantir une échelle comparable entre les variables pour les modèles sensibles aux amplitudes des données.

Le choix du scaler est effectué à partir des résultats de l'analyse exploratoire. Les boxplots des variables numériques n'ont mis en évidence aucun outlier significatif. En conséquence, StandardScaler est retenu pour standardiser les variables numériques. Si des valeurs aberrantes importantes avaient été observées, RobustScaler, basé sur la médiane et l'écart interquartile (IQR), aurait été privilégié afin de limiter leur influence.

Cette transformation est intégrée au pipeline de prétraitement (Pipeline et ColumnTransformer) afin d'être ajustée uniquement sur les données d'entraînement, puis appliquée de manière identique aux données de test et aux nouvelles observations, évitant ainsi toute fuite de données (data leakage).

# Train/Test Split

In [5]:
X = df.drop(columns=["LoyerMensuel_BIF"])
y = df["LoyerMensuel_BIF"]
y = np.array(y).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
# Creation du preprocessor
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist() 
# "category" pour inclure les catégories pandas

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Vérification explicite qu'aucune statistique n'a été calculée sur l'ensemble complet avant le split

Afin de garantir une évaluation réaliste des performances du modèle, la séparation entre les ensembles d'entraînement et de test est réalisée avant toute opération nécessitant l'estimation de paramètres à partir des données. Cette approche permet d'éviter tout phénomène de data leakage, où des informations provenant du jeu de test influenceraient indirectement l'entraînement du modèle.

Les données sont séparées selon une proportion de 80 % pour l'entraînement et 20 % pour le test à l'aide de train_test_split.

Toutes les transformations apprenantes (fit) sont ensuite réalisées exclusivement sur l'ensemble d'entraînement, notamment :

l'imputation des valeurs manquantes (SimpleImputer) ;
la standardisation des variables numériques (StandardScaler ou RobustScaler) ;
l'encodage de la variable catégorielle Quartier (OneHotEncoder).

Les paramètres ainsi estimés sont ensuite appliqués (transform) à l'ensemble de test, sans être recalculés. Cette stratégie garantit que les performances mesurées reflètent fidèlement la capacité de généralisation du modèle sur de nouvelles données.